# DPA News Indices

This notebook loads raw DPA sentences and BERT-labeled sentence files, applies basic filtering, assigns each sentence to a survey period (15th-of-month) or press-conference window, aggregates to monthly shares by label, and exports the resulting indices to Excel.


In [10]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent
DATA_DIR  = REPO_ROOT / "Data"
DPA_DIR   = DATA_DIR / "dpa"

PATH   = DATA_DIR / "Regression"
OUTDIR = PATH / "Neu"
OUTDIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Load raw sentence data

data = pd.read_csv(
    DPA_DIR / "dpa_prepro_final_sentences_no_lemmas_no_tokens.csv",
    encoding="utf-8",
    keep_default_na=False,
    dtype={"title": "str", "texts": "str"},
    usecols=["date", "title", "texts"],
)

data_new = pd.read_csv(
    DPA_DIR / "dpa_prepro_final_sentences_no_lemmas_full_with_lemmas_new_dpa.csv",
    encoding="utf-8",
    keep_default_na=False,
    dtype={"title": "object", "texts": "object"},
    usecols=["date", "title", "texts"],
    low_memory=False,
)

data = pd.concat([data, data_new], ignore_index=True)

In [3]:
# Load labeled sentence data

def load_labeled(path_old, path_new):
    df0 = pd.read_excel(path_old, usecols=["tokens", "date", "text", "Label"])
    df1 = pd.read_excel(path_new, usecols=["tokens", "date", "text", "Label"])
    df = pd.concat([df0, df1], ignore_index=True)
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_localize(None)
    return df

data_inf_all = load_labeled(
    DPA_DIR / "sent_inf_label_BERTv3.xlsx",
    DPA_DIR / "sent_inf_label_BERTv3_new_dpa.xlsx",
)
data_sent_all = load_labeled(
    DPA_DIR / "sent_sent_label_BERTv3.xlsx",
    DPA_DIR / "sent_sent_label_BERTv3_new_dpa.xlsx",
)
data_mon_all = load_labeled(
    DPA_DIR / "sent_mon_label_BERTv3.xlsx",
    DPA_DIR / "sent_mon_label_BERTv3_new_dpa.xlsx",
)
data_sentmon_all = load_labeled(
    DPA_DIR / "sent_sentmon_label_BERTv3.xlsx",
    DPA_DIR / "sent_sentmon_label_BERTv3_new_dpa.xlsx",
)

data["date"] = pd.to_datetime(data["date"], errors="coerce", utc=True).dt.tz_localize(None)

In [4]:
## Apply cutoff + drop “Referenzkurs” sentences

cutoff_start = pd.Timestamp("2001-11-15")
cutoff_end   = pd.Timestamp("2023-12-15")

def apply_filters_all_sent(df):
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce", utc=True).dt.tz_localize(None)
    out = out.loc[out["date"].between(cutoff_start, cutoff_end, inclusive="both")].copy()
    out = out[~out["texts"].astype(str).str.contains("Referenzkurs", case=False, na=False)]
    return out

def apply_filters_labeled(df):
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce", utc=True).dt.tz_localize(None)
    out = out.loc[out["date"].between(cutoff_start, cutoff_end, inclusive="both")].copy()
    out = out[~out["text"].astype(str).str.contains("Referenzkurs", case=False, na=False)]
    return out

data = apply_filters_all_sent(data)
data_inf_all = apply_filters_labeled(data_inf_all)
data_sent_all = apply_filters_labeled(data_sent_all)
data_mon_all = apply_filters_labeled(data_mon_all)
data_sentmon_all = apply_filters_labeled(data_sentmon_all)

In [7]:
# Assign periods

version = "survey"  # "press_conference" or "survey"

def compute_periods(df, version="survey"):
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce", utc=True).dt.tz_localize(None)

    if version == "survey":
        agg_dates = pd.date_range(start="2001-11-01", periods=266, freq="MS") + pd.Timedelta(days=14)
        a = pd.to_datetime(agg_dates).normalize().to_numpy()
        d = out["date"].dt.normalize().to_numpy()

        idx = np.searchsorted(a, d, side="left")
        idx = np.clip(idx, 0, len(a) - 1)

        out["Period"] = idx
        out["t_date"] = out["Period"].map({i: pd.Timestamp(dt) for i, dt in enumerate(agg_dates)})
        return out

    if version == "press_conference":
        press_sents = pd.read_excel(DATA_DIR / "ECB_sents_prepared.xlsx")
        press_sents["date"] = pd.to_datetime(press_sents["date"], errors="coerce", utc=True).dt.tz_localize(None)

        agg_dates = np.sort(press_sents["date"].dropna().dt.normalize().unique())
        a = pd.to_datetime(agg_dates).to_numpy()
        d = out["date"].dt.normalize().to_numpy()

        prev_idx = np.searchsorted(a, d, side="right") - 1
        prev_idx = np.clip(prev_idx, 0, len(a) - 1)

        delta_days = ((d - a[prev_idx]) / np.timedelta64(1, "D")).astype(float)
        inwin = np.isfinite(delta_days) & (delta_days >= 0) & (delta_days <= 14)

        out = out[inwin].copy()
        out["Period"] = prev_idx[inwin]
        out["t_date"] = out["Period"].map({i: pd.Timestamp(dt) for i, dt in enumerate(agg_dates)})
        return out

    raise ValueError("version must be 'survey' or 'press_conference'")

In [8]:
data = compute_periods(data, version)
data_inf_all = compute_periods(data_inf_all, version)
data_sent_all = compute_periods(data_sent_all, version)
data_mon_all = compute_periods(data_mon_all, version)
data_sentmon_all = compute_periods(data_sentmon_all, version)

In [9]:
# Compute News indices

def monthly_counts(df):
    s = df.groupby("t_date").size()
    s.index = pd.to_datetime(s.index)
    s = s.resample("M").ffill()
    all_months = pd.date_range("1999-12-31", "2023-12-31", freq="M")
    return s.reindex(all_months, fill_value=0)

def news_indices(data_inf, data_mon, data_sent, data_all):
    data_count = monthly_counts(data_all)

    rising  = monthly_counts(data_inf[data_inf["Label"] == 2]) / data_count
    notrend = monthly_counts(data_inf[data_inf["Label"] == 1]) / data_count
    falling = monthly_counts(data_inf[data_inf["Label"] == 0]) / data_count

    hawkish = (monthly_counts(data_mon[data_mon["Label"] == 2]) / data_count).fillna(0)
    nomon   = (monthly_counts(data_mon[data_mon["Label"] == 1]) / data_count).fillna(0)
    dovish  = (monthly_counts(data_mon[data_mon["Label"] == 0]) / data_count).fillna(0)

    good    = monthly_counts(data_sent[data_sent["Label"] == 2]) / data_count
    neutral = monthly_counts(data_sent[data_sent["Label"] == 1]) / data_count
    bad     = monthly_counts(data_sent[data_sent["Label"] == 0]) / data_count

    inf_number = monthly_counts(data_inf) / data_count
    mon_number = monthly_counts(data_mon) / data_count

    out = {
        "news_index_rising": rising.fillna(0),
        "news_index_notrend": notrend.fillna(0),
        "news_index_falling": falling.fillna(0),
        "news_index_hawkish": hawkish.fillna(0),
        "news_index_nomon": nomon.fillna(0),
        "news_index_dovish": dovish.fillna(0),
        "news_index_good": good.fillna(0),
        "news_index_neutral": neutral.fillna(0),
        "news_index_bad": bad.fillna(0),
        "news_index_inf_number": inf_number.fillna(0),
        "news_index_mon_number": mon_number.fillna(0),
        "data_count": data_count,
    }
    return out

news = news_indices(data_inf_all, data_mon_all, data_sent_all, data)


In [ ]:
news["news_index_rising"].to_excel(OUTDIR / "news_index_rising.xlsx")
news["news_index_notrend"].to_excel(OUTDIR / "news_index_notrend.xlsx")
news["news_index_falling"].to_excel(OUTDIR / "news_index_falling.xlsx")

news["news_index_hawkish"].to_excel(OUTDIR / "news_index_hawkish.xlsx")
news["news_index_nomon"].to_excel(OUTDIR / "news_index_nomon.xlsx")
news["news_index_dovish"].to_excel(OUTDIR / "news_index_dovish.xlsx")

news["news_index_good"].to_excel(OUTDIR / "news_index_good.xlsx")
news["news_index_neutral"].to_excel(OUTDIR / "news_index_neutral.xlsx")
news["news_index_bad"].to_excel(OUTDIR / "news_index_bad.xlsx")

news["news_index_inf_number"].to_excel(OUTDIR / "news_index_inf_number.xlsx")
news["news_index_mon_number"].to_excel(OUTDIR / "news_index_mon_number.xlsx")